# BitRoss 2 · 3/3 · Train

Trains the masked-diffusion pixel transformer on the curated cache. Checkpoints go to Drive every
15 minutes and training resumes automatically, so a disconnect only costs a few minutes.

**Runtime:** L4 (the default `small_l4` config) or A100 (`small_a100` / `base_a100`).
Sample grids are written every 2,000 steps to `RUN_DIR/samples`.

In [ ]:
#@title Setup: Drive, code, Hugging Face login
#@markdown Put your Hugging Face token in Colab **Secrets** (key icon in the left bar) as `HF_TOKEN`
#@markdown and enable notebook access. The dataset is private, so the token is required.
REPO_URL = "https://github.com/OVAWARE/BitRoss"  #@param {type:"string"}
BRANCH = "v2"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/BitRoss2"  #@param {type:"string"}

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
if not os.path.exists("/content/BitRoss"):
    code_zip = f"{DRIVE_ROOT}/code.zip"
    if os.path.exists(code_zip):            # fallback: upload a zip of the repo to Drive
        subprocess.run(["unzip", "-q", code_zip, "-d", "/content/BitRoss"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, "/content/BitRoss"], check=True)
os.chdir("/content/BitRoss")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!pip -q install -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title Copy the cache to local disk
!mkdir -p /content/cache && cp -r {DRIVE_ROOT}/cache/* /content/cache/
!cat /content/cache/info.json

In [ ]:
#@title Train
CONFIG = "configs/small_l4.json"  #@param ["configs/small_l4.json", "configs/small_a100.json", "configs/base_a100.json", "configs/nano_debug.json"]
RUN_NAME = "bitross-small"  #@param {type:"string"}
RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}"
!python -m bitross.train --config {CONFIG} --data /content/cache --out {RUN_DIR}

In [ ]:
#@title Latest samples and validation curves
import json, glob
from IPython.display import Image, display
rows = [json.loads(l) for l in open(f"{RUN_DIR}/metrics.jsonl")]
ev = [r for r in rows if "val_nll@0.5" in r]
for r in ev[-5:]:
    print(r["step"], {k: round(v, 4) for k, v in r.items() if k.startswith("val_")})
display(Image(sorted(glob.glob(f"{RUN_DIR}/samples/*.png"))[-1]))

In [ ]:
#@title Try it: launch BitRoss Studio with a public link
!python app.py --model {RUN_DIR}/bitross.pt --share

In [ ]:
#@title (Optional) publish the model to the Hugging Face Hub
REPO_ID = "OVAWARE/BitRoss-2"  #@param {type:"string"}
PRIVATE = True  #@param {type:"boolean"}
!python -m bitross.publish --run {RUN_DIR} --repo {REPO_ID} {"--private" if PRIVATE else ""}